<a href="https://colab.research.google.com/github/HadiAkbar/bookwise-recommender/blob/main/BookWise%E2%80%93Content_Based_Book_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---
<center>

**BookWise – Content-Based Book Recommender**

**CS 4403 - Data Mining**

**Hadi Akbar**


---



In [ ]:
# Importing the required libraries
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

---
#Task 1
#Loading and exploration of the Data

---

In [ ]:
books = [
    {"id": 1, "title": "Dune", "genre": "Sci-Fi", "description": "A desert planet holds the most precious resource in the universe. A noble family is drawn into a deadly political and religious conflict over its control."},

    {"id": 2, "title": "Foundation","genre": "Sci-Fi", "description": "A mathematician predicts the fall of civilization and secretly plans to shorten the coming dark age through a long-term scientific project."},

    {"id": 3, "title": "Neuromancer", "genre": "Sci-Fi", "description": "A washed-up hacker is hired for one last heist in a neon-lit future of corporate espionage, artificial intelligence, and cyberspace."},

    {"id": 4, "title": "The Name of the Wind", "genre": "Fantasy", "description": "A legendary wizard recounts his extraordinary life, from a childhood among traveling performers to his years at a magical university."},

    {"id": 5, "title": "The Way of Kings", "genre": "Fantasy", "description": "On a world ravaged by storms, a slave, a scholar, and a reluctant warrior are each drawn into an ancient conflict that will decide humanity's fate."},

    {"id": 6, "title": "The Hobbit", "genre": "Fantasy", "description": "A reluctant homebody is swept into a grand adventure with a company of dwarves seeking to reclaim their mountain home from a fearsome dragon."},

    {"id": 7, "title": "The Martian", "genre": "Sci-Fi","description": "An astronaut is accidentally stranded on Mars and must use ingenuity and dark humor to survive until a rescue mission can reach him."},

    {"id": 8, "title": "Ender's Game", "genre": "Sci-Fi", "description": "A child prodigy is trained at a remote battle school to become the commander Earth needs to defeat an alien invasion."},

    {"id": 9, "title": "A Wizard of Earthsea", "genre": "Fantasy", "description": "A young boy with raw magical talent enrolls in a school for wizards and must hunt down the shadow creature he accidentally unleashed."},

    {"id": 10, "title": "Recursion", "genre": "Thriller", "description": "A neuroscientist and a police detective uncover a memory-altering technology that is quietly rewriting people's pasts and destabilizing reality."},

    {"id": 11, "title": "Gone Girl", "genre": "Thriller", "description": "When a woman vanishes on her anniversary, her charming husband becomes the prime suspect, but both have been keeping dangerous secrets."},

    {"id": 12, "title": "The Girl with the Dragon Tattoo", "genre": "Thriller", "description": "A disgraced journalist and a brilliant hacker investigate a decades-old disappearance within a wealthy and deeply dysfunctional Swedish family."},
]

df = pd.DataFrame(books)

df.head(12)

,id,title,genre,description
0,1,Dune,Sci-Fi,A desert planet holds the most precious resour...
1,2,Foundation,Sci-Fi,A mathematician predicts the fall of civilizat...
2,3,Neuromancer,Sci-Fi,A washed-up hacker is hired for one last heist...
3,4,The Name of the Wind,Fantasy,A legendary wizard recounts his extraordinary ...
4,5,The Way of Kings,Fantasy,"On a world ravaged by storms, a slave, a schol..."
5,6,The Hobbit,Fantasy,A reluctant homebody is swept into a grand adv...
6,7,The Martian,Sci-Fi,An astronaut is accidentally stranded on Mars ...
7,8,Ender's Game,Sci-Fi,A child prodigy is trained at a remote battle ...
8,9,A Wizard of Earthsea,Fantasy,A young boy with raw magical talent enrolls in...
9,10,Recursion,Thriller,A neuroscientist and a police detective uncove...


In [ ]:
# Count books per genre
genre_counts = df['genre'].value_counts()
print(genre_counts)

genre
Sci-Fi      5
Fantasy     4
Thriller    3
Name: count, dtype: int64


---
<center>

**Genre Distribution**

---

The dataset consists of 12 books categorized into three genres:

Sci-Fi: 5 books

Fantasy: 4 books

Thriller: 3 books

Sci-Fi is the most common genre in this dataset followed by Fantasy and then Thriller.

---
<center>

**Books I expect to be similar**

---
**The Name of the Wind** and **A Wizard of Earthsea**

Both stories focus on young individuals with magical abilities who attend a form of magical training or school. They share themes of personal growth, learning magic, and facing the consequences of power.

---
<center>

**Books I expect to be different**

---
**The Hobbit** and **Neuromancer**

The Hobbit is a classic fantasy adventure involving a quest, mythical creatures, and a medieval style world, while Neuromancer is a cyberpunk science fiction story centered on hacking, artificial intelligence, and a futuristic digital world. Their settings, themes, and tones are completely different.

---
#Task 2
# Embeddings

---

In [ ]:
# Load the all-MiniLM model which is given in the assignment
model1 = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings1 = model1.encode(df['description'].tolist())

print("Embedding shape (Model 1 - all-MiniLM-L6-v2):", embeddings1.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape (Model 1 - all-MiniLM-L6-v2): (12, 384)


---
<center>

**Questions**

---

**Research the model used. How was it created, and what is it good at?**

**Response:** The model we used is called <mark>"all-MiniLM-L6-v2 model"</mark>, which is a sentencetransformers model. It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search.

The all-MiniLM-L6-v2 model was created as a high-performance, lightweight sentence-transformer, trained during a Hugging Face community week on over 1 billion sentence pairs. Using contrastive learning (specifically Multiple Negative Ranking Loss), it was designed to map sentences and paragraphs into a 384-dimensional dense vector space for tasks like semantic search and clustering, providing efficient, high-quality embeddings. (https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)

The model is good at Semantic similarity tasks, Text clustering, Information retrieval and Recommendation systems.

---

**What is the shape of the embeddings array?**

**Response:** As we have 12 books and the model maps into 384 dimensions vector so the the shape of the embeddings array would be (12, 384).

---
**What does each dimension represent?**

**Response:** According to the models documentation each dimension represents a learned, abstract feature capturing semantic, syntactic, or contextual information about the input text.

---
**The model name all-MiniLM-L6-v2 is a Sentence-BERT variant.**

**What does “sentence-level” embedding mean, as opposed to “word-level” embedding?**

**Response:** A sentence level embedding means the conversion of an entire sentence or paragraph as a single vector.

Whereas word level embedding assigns vectors to individual words. Sentence level embedding captures the overall meaning and context of the text instead of just individual words.

---

In [ ]:
# Loading the second model of my choice
model_nomic = SentenceTransformer('nomic-ai/nomic-embed-text-v1', trust_remote_code=True)

embeddings_nomic = model_nomic.encode(df['description'].tolist())

print("Embedding shape (Model 2 - Nomic):", embeddings_nomic.shape)

Embedding shape (Model 2 - Nomic): (12, 768)


---
<center>

**Second Model**

<mark>**NOMIC v1**</mark>

---

**Why did I choose this model?**

**Response:**

![Alt text](https://supermemory.ai/blog/content/images/2025/06/Embedding_Models.webp)

From my reasearch Nomic is among the best open soure models out there. It is a modern and high performance embedding model that has shown strong results on semantic similarity and clustering benchmarks. For its comparison to other models I have attached the graphical illustration.

The reason of me choosing NOMIC was that I wanted to experiment with a more recent model beyond the traditional models to see whether it improves recommendation quality.

The embedding array dimension for Nomic-V1 is (12, 768).

In [ ]:
# Saving the embeddings
# Model 1 (all-MiniLM-L6-v2)
model1 = SentenceTransformer('all-MiniLM-L6-v2')
embeddings_minilm = model1.encode(df['description'].tolist())

# Model 2 (Nomic-v1)
model_nomic = SentenceTransformer('nomic-ai/nomic-embed-text-v1', trust_remote_code=True)
embeddings_nomic = model_nomic.encode(df['description'].tolist())

print("MiniLM shape:", embeddings_minilm.shape)
print("Nomic shape:", embeddings_nomic.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM shape: (12, 384)
Nomic shape: (12, 768)


In [ ]:
# Normalizing the embeddings
from sklearn.preprocessing import normalize

embeddings_minilm_norm = normalize(embeddings_minilm)
embeddings_nomic_norm = normalize(embeddings_nomic)

I generated embeddings using both the models.

all-MiniLM-L6-v2 embeddings are stored in embeddings_minilm.

Nomic embeddings are stored in embeddings_nomic.

I alse created normalized versions of the embeddings (embeddings_minilm_norm and embeddings_nomic_norm) to improve cosine similarity calculations.

---
#Task 3

#Measure Semantic Similarity

---

In [ ]:
def compute_similarity_matrix(embeddings):
    sim_matrix = cosine_similarity(embeddings)
    return np.round(sim_matrix, 3)

In [ ]:
sim_minilm = compute_similarity_matrix(embeddings_minilm_norm)
sim_nomic = compute_similarity_matrix(embeddings_nomic_norm)

print("MiniLM Similarity Matrix:\n", sim_minilm)
print("\nNomic Similarity Matrix:\n", sim_nomic)

MiniLM Similarity Matrix:
 [[ 1.     0.152  0.008  0.039  0.442  0.143  0.233  0.14   0.053 -0.013
   0.123  0.055]
 [ 0.152  1.     0.32   0.195  0.391  0.124  0.268  0.19   0.141  0.343
   0.208  0.284]
 [ 0.008  0.32   1.     0.111  0.175  0.074  0.246  0.225  0.183  0.382
   0.172  0.445]
 [ 0.039  0.195  0.111  1.     0.209  0.199  0.146  0.191  0.458  0.188
   0.065  0.135]
 [ 0.442  0.391  0.175  0.209  1.     0.266  0.133  0.243  0.187  0.189
   0.121  0.181]
 [ 0.143  0.124  0.074  0.199  0.266  1.     0.111  0.125  0.16   0.059
   0.07   0.092]
 [ 0.233  0.268  0.246  0.146  0.133  0.111  1.     0.124  0.162  0.164
   0.178  0.205]
 [ 0.14   0.19   0.225  0.191  0.243  0.125  0.124  1.     0.515  0.186
  -0.082  0.041]
 [ 0.053  0.141  0.183  0.458  0.187  0.16   0.162  0.515  1.     0.212
   0.01   0.127]
 [-0.013  0.343  0.382  0.188  0.189  0.059  0.164  0.186  0.212  1.
   0.303  0.397]
 [ 0.123  0.208  0.172  0.065  0.121  0.07   0.178 -0.082  0.01   0.303
   1.     0.35

---
<center>

**LLM Used**

---

I consulted muitipled LLMs and leaned towards using the response of ChatGPT to help generate the code for computing the cosine similarity matrix.

---
**Prompt Used**

---
"write me a python function which i can use that computes the full pair wise cosine similarity matrix for a set of embeddings and it should round the result to 3 decimal places no more no less."

---

In [ ]:
# Highest and lowest similarity
def find_extremes(sim_matrix, df):
    n = len(sim_matrix)

    max_val = -1
    min_val = 2
    max_pair = None
    min_pair = None

    for i in range(n):
        for j in range(i+1, n):
            if sim_matrix[i][j] > max_val:
                max_val = sim_matrix[i][j]
                max_pair = (df.iloc[i]['title'], df.iloc[j]['title'])

            if sim_matrix[i][j] < min_val:
                min_val = sim_matrix[i][j]
                min_pair = (df.iloc[i]['title'], df.iloc[j]['title'])

    return max_pair, max_val, min_pair, min_val


# MiniLM results
max_pair, max_val, min_pair, min_val = find_extremes(sim_minilm, df)

print("MiniLM Results")
print("Highest Similarity:", max_pair, "Score:", round(float(max_val), 3))
print("Lowest Similarity:", min_pair, "Score:", round(float(min_val), 3))


# Nomic results
max_pair, max_val, min_pair, min_val = find_extremes(sim_nomic, df)

print("\nNomic Results")
print("Highest Similarity:", max_pair, "Score:", round(float(max_val), 3))
print("Lowest Similarity:", min_pair, "Score:", round(float(min_val), 3))

MiniLM Results
Highest Similarity: ("Ender's Game", 'A Wizard of Earthsea') Score: 0.515
Lowest Similarity: ("Ender's Game", 'Gone Girl') Score: -0.082

Nomic Results
Highest Similarity: ("Ender's Game", 'A Wizard of Earthsea') Score: 0.665
Lowest Similarity: ("Ender's Game", 'Gone Girl') Score: 0.352


---
<center>

**Highest Similarity Pair**

---

**MiniLM Model**

*Ender’s Game* and *A Wizard of Earthsea*  
Score: **0.515**

**Nomic Model**

*Ender’s Game* and *A Wizard of Earthsea*  
Score: **0.665**

---

**Do these books make intuitive sense as similar?**

---

Initially the result seems unexpected because the books belong to different genres (science fiction and fantasy). But it does make sense when we consider deeper similarities.

Like as both of the books feature young protagonists with exceptional abilities and focus on their development through structured training environments. They also explore themes of growth, responsibility, and mastering power. This shows that the models are capturing semantic meaning not only the genre.

---

**Lowest Similarity Pair**

---

**MiniLM Model**

*Ender’s Game* and *Gone Girl*  
Score: **-0.082**

**Nomic Model**

*Ender’s Game* and *Gone Girl*  
Score: **0.352**

---

**Are you surprised?**

---

The MiniLM result is not surprising as the books are very different in both genre and theme. One focuses on futuristic space warfare, while the other is a psychological thriller centered on relationships and crime.

The Nomic result is somewhat surprising because it still shows a moderate similarity score of 0.0352. This may indicate that the model captures more abstract similarities such as psychological tension or strategic thinking, even across different genres.

---

**My Predictions**

---

Most similar: *The Name of the Wind* and *A Wizard of Earthsea*  
Most different: *The Hobbit* and *Neuromancer*

---

**Was I correct?**

---

No.... I was not correct.

---

**Where did my intuition diverge from the models' scores?**

---

My predictions were mainly based on genre and obvious themes. The models identified deeper similarities such as character roles and narrative structure. This demonstrates that embeddings capture latent semantic relationships that are not immediately obvious to a humane way of thinking.

---

**What is always true about the diagonal of the similarity matrix, and why?**

---

The diagonal values of the similarity matrix are always **1**.

Each book is compared with itself and the cosine similarity of a vector with itself is always 1 representing perfect similarity.

---

**LLM Used**

---

I used ChatGPT to understand the properties of the similarity matrix.

---

**Prompt Used**

---

"What is always true about the diagonal of a cosine similarity matrix and why?"

---

---
#Task 4

#Build the Recommender

---

In [ ]:
# Recommender function
def recommend_books(title, df, sim_matrix, k=3, genre=None):
    idx = df[df['title'] == title].index[0]
    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = []

    for i, score in scores:
        if i == idx:
            continue

        if genre is not None and df.iloc[i]['genre'] != genre:
            continue

        recommendations.append((df.iloc[i]['title'], round(float(score), 3)))

        if len(recommendations) == k:
            break

    return recommendations


# print
def print_recommendations(title, recs, model_name):
    print(f"\n{model_name} - Recommendations for '{title}':")
    for i, (book, score) in enumerate(recs, 1):
        print(f"{i}. {book} (Score: {score})")

# MiniLM Recommendations
print("MiniLM Recommendations:")

print_recommendations("Dune", recommend_books("Dune", df, sim_minilm), "MiniLM")
print_recommendations("The Hobbit", recommend_books("The Hobbit", df, sim_minilm), "MiniLM")
print_recommendations("Gone Girl", recommend_books("Gone Girl", df, sim_minilm), "MiniLM")

# Nomic Recommendations
print("\nNomic Recommendations:")

print_recommendations("Dune", recommend_books("Dune", df, sim_nomic), "Nomic")
print_recommendations("The Hobbit", recommend_books("The Hobbit", df, sim_nomic), "Nomic")
print_recommendations("Gone Girl", recommend_books("Gone Girl", df, sim_nomic), "Nomic")

# Genre Filter Example
print("\nWith Genre Filter (Sci-Fi):")

filtered_minilm = recommend_books("Dune", df, sim_minilm, genre="Sci-Fi")
print_recommendations("Dune", filtered_minilm, "MiniLM")

filtered_recs = recommend_books("Dune", df, sim_nomic, genre="Sci-Fi")
print_recommendations("Dune", filtered_recs, "Nomic")

MiniLM Recommendations:

MiniLM - Recommendations for 'Dune':
1. The Way of Kings (Score: 0.442)
2. The Martian (Score: 0.233)
3. Foundation (Score: 0.152)

MiniLM - Recommendations for 'The Hobbit':
1. The Way of Kings (Score: 0.266)
2. The Name of the Wind (Score: 0.199)
3. A Wizard of Earthsea (Score: 0.16)

MiniLM - Recommendations for 'Gone Girl':
1. The Girl with the Dragon Tattoo (Score: 0.358)
2. Recursion (Score: 0.303)
3. Foundation (Score: 0.208)

Nomic Recommendations:

Nomic - Recommendations for 'Dune':
1. The Way of Kings (Score: 0.617)
2. The Hobbit (Score: 0.533)
3. Ender's Game (Score: 0.53)

Nomic - Recommendations for 'The Hobbit':
1. A Wizard of Earthsea (Score: 0.554)
2. Dune (Score: 0.533)
3. The Way of Kings (Score: 0.525)

Nomic - Recommendations for 'Gone Girl':
1. The Girl with the Dragon Tattoo (Score: 0.598)
2. Recursion (Score: 0.534)
3. Foundation (Score: 0.463)

With Genre Filter (Sci-Fi):

MiniLM - Recommendations for 'Dune':
1. The Martian (Score: 0.23

---

### **MiniLM Recommendations**

**Dune:**
1. The Way of Kings (Score: 0.442)
2. The Martian (Score: 0.233)
3. Foundation (Score: 0.152)

**The Hobbit:**
1. The Way of Kings (Score: 0.266)
2. The Name of the Wind (Score: 0.199)
3. A Wizard of Earthsea (Score: 0.16)

**Gone Girl:**
1. The Girl with the Dragon Tattoo (Score: 0.358)
2. Recursion (Score: 0.303)
3. Foundation (Score: 0.208)

---

### **Nomic Recommendations**

**Dune:**
1. The Way of Kings (Score: 0.617)
2. The Hobbit (Score: 0.533)
3. Ender's Game (Score: 0.53)

**The Hobbit:**
1. A Wizard of Earthsea (Score: 0.554)
2. Dune (Score: 0.533)
3. The Way of Kings (Score: 0.525)

**Gone Girl:**
1. The Girl with the Dragon Tattoo (Score: 0.598)
2. Recursion (Score: 0.534)
3. Foundation (Score: 0.463)

---

### **With Genre Filter (Sci-Fi)**

**MiniLM - Dune:**
1. The Martian (Score: 0.233)
2. Foundation (Score: 0.152)
3. Ender's Game (Score: 0.14)

**Nomic - Dune:**
1. Ender's Game (Score: 0.53)
2. The Martian (Score: 0.505)
3. Foundation (Score: 0.492)

---

---
<center>

**Are the recommendations always from the same genre as the query?**

---

No, the recommendations are not always from the same genre. The system is content based and relies on semantic similarity of descriptions not the genre label. This means that two books from different genres can still be recommended if their themes, plot structures, or character types are similar.  

For example: MiniLM recommended **The Martian** (Sci-Fi) for **Dune** (Sci-Fi) and Nomic recommended **Ender's Game** (Sci-Fi) for **Dune**. But some books like **A Wizard of Earthsea** (Fantasy) also appeared as similar to other genres because of narrative and theme alike.

---

**How to filter by genre?**

---

It is already implemented in the code above but for demontration purpose I will implement it here again. The way it is done is that the function includes a genre parameter that restricts recommendations to a specific genre.

In [ ]:
recommend_books("Dune", df, sim_nomic, genre="Sci-Fi")

[("Ender's Game", 0.53), ('The Martian', 0.505), ('Foundation', 0.492)]

<center>

---

**Differences**

---

Yes, there are differences:

MiniLM tends to return slightly lower similarity scores and more conservative recommendations.

Nomic gives higher similarity scores and may capture more abstract or thematic similarities beyond surface level description.

These differences exist because the models are trained differently and have varying capacity to capture semantics. Nomic embeddings are generally more powerful for nuanced semantic relationships, while MiniLM is lighter and faster.

---

<center>

---

**User's History based recommendations**

---

In [ ]:
# Function to recommend books for a user based on multiple liked books
# Adds model name to the output
def recommend_for_user(user_name, liked_books, df, embeddings, k=3, genre=None, model_name="Model"):
    # get indices of liked books
    indices = [df[df['title'] == book].index[0] for book in liked_books]

    # compute average embedding vector for user profile
    user_vector = np.mean(embeddings[indices], axis=0, keepdims=True)

    # cosine similarity between user vector and all books
    sim_scores = cosine_similarity(user_vector, embeddings).flatten()

    # sort scores in descending order
    scores = list(enumerate(sim_scores))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = []

    for i, score in scores:
        # skip books user already likes
        if df.iloc[i]['title'] in liked_books:
            continue

        # filter by genre if specified
        if genre is not None and df.iloc[i]['genre'] != genre:
            continue

        recommendations.append((df.iloc[i]['title'], round(float(score), 3)))

        if len(recommendations) == k:
            break

    # print recommendations neatly with model name
    print(f"\n{model_name} - Recommendations for {user_name} based on liked books: {liked_books}")
    for idx, (book, score) in enumerate(recommendations, 1):
        print(f"{idx}. {book} (Score: {score})")

    return recommendations

In [ ]:
# Hadi likes Dune and The Hobbit
liked_books_hadi = ["Dune", "The Hobbit"]

# MiniLM recommendations
recommend_for_user("Hadi", liked_books_hadi, df, embeddings_minilm_norm, k=5, model_name="MiniLM")

# Nomic recommendations
recommend_for_user("Hadi", liked_books_hadi, df, embeddings_nomic_norm, k=5, model_name="Nomic")


MiniLM - Recommendations for Hadi based on liked books: ['Dune', 'The Hobbit']
1. The Way of Kings (Score: 0.468)
2. The Martian (Score: 0.228)
3. Foundation (Score: 0.182)
4. Ender's Game (Score: 0.175)
5. The Name of the Wind (Score: 0.157)

Nomic - Recommendations for Hadi based on liked books: ['Dune', 'The Hobbit']
1. The Way of Kings (Score: 0.652)
2. Ender's Game (Score: 0.594)
3. A Wizard of Earthsea (Score: 0.58)
4. The Martian (Score: 0.571)
5. Foundation (Score: 0.553)


[('The Way of Kings', 0.652),
 ("Ender's Game", 0.594),
 ('A Wizard of Earthsea', 0.58),
 ('The Martian', 0.571),
 ('Foundation', 0.553)]

<center>

---

**MiniLM - Recommendations for Hadi based on liked books: ['Dune', 'The Hobbit']**

---
1. The Way of Kings (Score: 0.468)
2. The Martian (Score: 0.228)
3. Foundation (Score: 0.182)
4. Ender's Game (Score: 0.175)
5. The Name of the Wind (Score: 0.157)

---

**Nomic - Recommendations for Hadi based on liked books: ['Dune', 'The Hobbit']**

---
1. The Way of Kings (Score: 0.652)
2. Ender's Game (Score: 0.594)
3. A Wizard of Earthsea (Score: 0.58)
4. The Martian (Score: 0.571)
5. Foundation (Score: 0.553)

---

**How I modified the system for a user’s entire reading history**

---

My recommender takes a **single book** as input and finds the most similar books based on its embedding. To handle **multiple liked books** (a user’s reading history), I created a **user profile embedding**:

1. Collected the embeddings of all the books the user has liked.
2. Average these embeddings to form a single vector representing the user’s overall preferences.
3. Compute cosine similarity between this user vector and all books in the catalog.
4. Return the top-k books with the highest similarity scores, excluding the books the user has already read.

This allows the system to recommend books that are **closer to the user’s combined interests** rather than just a single book.

---

---
#Task 5

#Embed Your Own Text

---

<center>

---
**My Prompt (Chatgpt)**

---

i need a 3 sentence original description for each of these books and make it concise and suitable for a recommendation system that I build the notebook is attached for you to go through the books are Harry Potter and the Sorcerer’s Stone, The Silent Patient, Sapiens: A Brief History of Humankind

---
**ChatGPT Response**

---

1. Harry Potter and the Sorcerer’s Stone: A young boy discovers he is a wizard and attends a magical school, uncovering secrets about his past and facing dark forces that threaten the wizarding world.  
2. The Silent Patient: A renowned painter shoots her husband and never speaks again; a psychotherapist becomes obsessed with uncovering the truth behind her silence.  
3. Sapiens: A Brief History of Humankind: A sweeping exploration of human history, tracing the evolution of Homo sapiens from hunter-gatherers to modern civilization and examining how culture, science, and society shape humanity.

---

In [ ]:
# New book descriptions
new_books = [
    "A young boy discovers he is a wizard and attends a magical school, uncovering secrets about his past and facing dark forces that threaten the wizarding world.",
    "A renowned painter shoots her husband and never speaks again; a psychotherapist becomes obsessed with uncovering the truth behind her silence.",
    "A sweeping exploration of human history, tracing the evolution of Homo sapiens from hunter-gatherers to modern civilization and examining how culture, science, and society shape humanity."
]

# Embed with MiniLM
new_embeddings_minilm = model1.encode(new_books, normalize_embeddings=True)

# Embed with Nomic
new_embeddings_nomic = model_nomic.encode(new_books, normalize_embeddings=True)

In [ ]:
# MiniLM similarities
sim_new_minilm = cosine_similarity(new_embeddings_minilm, embeddings_minilm_norm)
print("MiniLM Similarities:\n", np.round(sim_new_minilm, 3))

# Nomic similarities
sim_new_nomic = cosine_similarity(new_embeddings_nomic, embeddings_nomic_norm)
print("\nNomic Similarities:\n", np.round(sim_new_nomic, 3))

MiniLM Similarities:
 [[0.04  0.322 0.158 0.57  0.254 0.136 0.133 0.403 0.74  0.306 0.145 0.194]
 [0.12  0.235 0.194 0.185 0.223 0.067 0.195 0.047 0.189 0.411 0.514 0.229]
 [0.17  0.417 0.209 0.259 0.441 0.275 0.129 0.138 0.125 0.304 0.099 0.296]]

Nomic Similarities:
 [[0.458 0.503 0.437 0.629 0.549 0.502 0.434 0.591 0.828 0.47  0.423 0.433]
 [0.417 0.422 0.381 0.367 0.424 0.337 0.383 0.357 0.366 0.553 0.601 0.487]
 [0.507 0.556 0.47  0.447 0.548 0.457 0.389 0.438 0.424 0.52  0.383 0.445]]


In [ ]:
# Titles of the new books
new_book_titles = [
    "Harry Potter and the Sorcerer's Stone",
    "The Silent Patient",
    "Sapiens: A Brief History of Humankind"
]

# Function to print readable top-k recommendations for new books with titles
def print_new_book_recommendations(new_books, new_embeddings, new_titles, df, model_name, top_k=3):
    sim_matrix = cosine_similarity(new_embeddings, embeddings_minilm_norm if model_name=='MiniLM' else embeddings_nomic_norm)

    for i, book_desc in enumerate(new_books):
        top_indices = sim_matrix[i].argsort()[::-1][:top_k]
        print(f"\n{model_name} - Recommendations for '{new_titles[i]}':")
        for rank, idx in enumerate(top_indices, start=1):
            print(f"{rank}. {df.iloc[idx]['title']} (Score: {sim_matrix[i][idx]:.3f})")

# MiniLM
print_new_book_recommendations(new_books, new_embeddings_minilm, new_book_titles, df, 'MiniLM')

# Nomic
print_new_book_recommendations(new_books, new_embeddings_nomic, new_book_titles, df, 'Nomic')


MiniLM - Recommendations for 'Harry Potter and the Sorcerer's Stone':
1. A Wizard of Earthsea (Score: 0.740)
2. The Name of the Wind (Score: 0.570)
3. Ender's Game (Score: 0.403)

MiniLM - Recommendations for 'The Silent Patient':
1. Gone Girl (Score: 0.514)
2. Recursion (Score: 0.411)
3. Foundation (Score: 0.235)

MiniLM - Recommendations for 'Sapiens: A Brief History of Humankind':
1. The Way of Kings (Score: 0.441)
2. Foundation (Score: 0.417)
3. Recursion (Score: 0.304)

Nomic - Recommendations for 'Harry Potter and the Sorcerer's Stone':
1. A Wizard of Earthsea (Score: 0.828)
2. The Name of the Wind (Score: 0.629)
3. Ender's Game (Score: 0.591)

Nomic - Recommendations for 'The Silent Patient':
1. Gone Girl (Score: 0.601)
2. Recursion (Score: 0.553)
3. The Girl with the Dragon Tattoo (Score: 0.487)

Nomic - Recommendations for 'Sapiens: A Brief History of Humankind':
1. Foundation (Score: 0.556)
2. The Way of Kings (Score: 0.548)
3. Recursion (Score: 0.520)


<center>

---
**Observations on Recommendations for My Own Books**

---

---
**MiniLM Recommendations:**

---

- *Harry Potter and the Sorcerer's Stone* is most similar to **A Wizard of Earthsea** and **The Name of the Wind** which makes sense because they all feature young wizards and magical education.  
- *The Silent Patient* is closest to **Gone Girl** and **Recursion**, both psychological/thriller-like stories with suspense and mystery.  
- *Sapiens: A Brief History of Humankind* is recommended with **The Way of Kings** and **Foundation**, which seems a little less intuitive probably because MiniLM captures high-level thematic or narrative complexity rather than strict genre.
---
**Nomic Recommendations:**

---

- *Harry Potter* recommendations are even stronger for fantasy titles (**A Wizard of Earthsea**, **The Name of the Wind**), reflecting Nomic’s more fine-grained semantic understanding.  
- *The Silent Patient* also matches thrillers (**Gone Girl**, **Recursion**, **The Girl with the Dragon Tattoo**) correctly.  
- *Sapiens* pairs best with **Foundation** and **The Way of Kings**, showing Nomic finds connections through world building, events, and conceptual similarity rather than only content type.

---

Both models successfully match books to similar themes, though Nomic tends to produce **higher similarity scores** and slightly more “conceptually meaningful” recommendations.  This approach confirms that text embeddings can be used to **extend recommendations beyond the original catalog**, which is a nice way to personalize the system. Human intuition roughly aligns with the results, but models can surprise you by finding **semantic connections** that are less obvious at first glance.

---

---
#Task 6

#Questions

---

---
**Q1: What information is captured by a sentence embedding that a simple bag-of-words TF-IDF vector would miss?**

---

**Response:** Sentence embeddings get the meaning and context of the text not just word counts so stuff like word order or synonyms matter for example "a wizard saves the kingdom" and "the magician protects the realm" would be far apart in tf-idf but close in embeddings.

---

**Q2: This system is content-based. What data would you need to build a collaborative filtering system instead, and what are the trade-offs between the two approaches?**  

---
**Response:** For collaborative filtering we need users item interactions like ratings, clicks or purchases. Content based only looks at descriptions. Collaborative can find patterns between users and items even if the content is different.
The trade off is that Collaborative filtering cannott handle new items well but finds taste patterns better whereas content based can handle new items but may be too focused on similar content.

---

**Q3: The model we used was pre-trained on general text. How might recommendations change if you fine-tuned it on a large corpus of book reviews?**

---

**Response:** If I fine tuned it on book reviews the embeddings would understand the books better like genre, writing style or sentiment so recommendations would probably be more accurate.

---

**Q4: Name one bias that could be introduced by using description-based embeddings, and suggest how you might mitigate it.**  

---

**Response:** Descriptions might overrepresent popular books or certain genres so those books could get recommended more. Mitigation could be to combine embeddings with user data or normalize description length or style.

---

**Q5: With millions of items, computing all pairwise cosine similarities becomes infeasible. Name one technique that addresses this scalability problem.**

---

**Response:** To address the scalability of millions of items we use Locality-Sensitive Hashing (LSH). This technique sorts items into "buckets" so that similar items likely end up in the same one. Instead of comparing everything we only calculate similarities for items sharing a bucket drastically reducing the workload.

---